In [73]:
import findspark
import pyspark
from pyspark.sql import SparkSession

findspark.init()
sc = pyspark.SparkContext.getOrCreate()
spark = SparkSession.builder.getOrCreate()

In [74]:
"""
RDD SOLUTION
"""

'\nRDD SOLUTION\n'

In [75]:
inputRDD = sc.textFile("data/readings.txt")
inputRDD.collect()

['1451606400,12.1',
 '1451606460,12.2',
 '1451606520,13.5',
 '1451606580,14.0',
 '1451606640,14.0',
 '1451606700,15.5',
 '1451606760,15.0']

In [76]:
def get_window(line):
    result = []
    for i in range(3):
        # I want to subtract 60 seconds from the timestamp,
        # in this way I can have a window of 3 values. I set
        # the new timestamp as the key of the RDD in order to
        # be able to group them and retrieve the values
        result.append((int(line.split(",")[0]) - i * 60, line))
    return result

In [77]:
# I apply flatMap to get all the possible combinations
mappedRDD = inputRDD.flatMap(get_window)
mappedRDD.collect()

[(1451606400, '1451606400,12.1'),
 (1451606340, '1451606400,12.1'),
 (1451606280, '1451606400,12.1'),
 (1451606460, '1451606460,12.2'),
 (1451606400, '1451606460,12.2'),
 (1451606340, '1451606460,12.2'),
 (1451606520, '1451606520,13.5'),
 (1451606460, '1451606520,13.5'),
 (1451606400, '1451606520,13.5'),
 (1451606580, '1451606580,14.0'),
 (1451606520, '1451606580,14.0'),
 (1451606460, '1451606580,14.0'),
 (1451606640, '1451606640,14.0'),
 (1451606580, '1451606640,14.0'),
 (1451606520, '1451606640,14.0'),
 (1451606700, '1451606700,15.5'),
 (1451606640, '1451606700,15.5'),
 (1451606580, '1451606700,15.5'),
 (1451606760, '1451606760,15.0'),
 (1451606700, '1451606760,15.0'),
 (1451606640, '1451606760,15.0')]

In [78]:
# I group the RDD by key in order to retrieve the values
groupedRDD = mappedRDD.groupByKey().map(lambda x: (x[0], list(x[1])))
groupedRDD.collect()

[(1451606400, ['1451606400,12.1', '1451606460,12.2', '1451606520,13.5']),
 (1451606340, ['1451606400,12.1', '1451606460,12.2']),
 (1451606280, ['1451606400,12.1']),
 (1451606460, ['1451606460,12.2', '1451606520,13.5', '1451606580,14.0']),
 (1451606520, ['1451606520,13.5', '1451606580,14.0', '1451606640,14.0']),
 (1451606580, ['1451606580,14.0', '1451606640,14.0', '1451606700,15.5']),
 (1451606640, ['1451606640,14.0', '1451606700,15.5', '1451606760,15.0']),
 (1451606700, ['1451606700,15.5', '1451606760,15.0']),
 (1451606760, ['1451606760,15.0'])]

In [79]:
def retrieve_lines(line):
    if len(line[1]) == 3:
        if float(line[1][0].split(",")[1]) < float(line[1][1].split(",")[1]) < float(line[1][2].split(",")[1]):
            return True
    return False

In [80]:
filteredRDD = groupedRDD.filter(retrieve_lines)
filteredRDD.collect()


[(1451606400, ['1451606400,12.1', '1451606460,12.2', '1451606520,13.5']),
 (1451606460, ['1451606460,12.2', '1451606520,13.5', '1451606580,14.0'])]